In [11]:
import torch
import torchvision
import os
import numpy as np
import matplotlib.pyplot as plt

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
Torchvision: 0.26.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [12]:
DATA_DIR = r"D:\Desktop\ml_lab\PlantVillage\PlantVillage"

print("Dataset exists:", os.path.exists(DATA_DIR))
print("Subfolders:", os.listdir(DATA_DIR)[:10])

Dataset exists: True
Subfolders: ['Pepper__bell___Bacterial_spot', 'Pepper__bell___healthy', 'Potato___Early_blight', 'Potato___healthy', 'Potato___Late_blight', 'Tomato_Bacterial_spot', 'Tomato_Early_blight', 'Tomato_healthy', 'Tomato_Late_blight', 'Tomato_Leaf_Mold']


In [13]:
from torchvision.datasets import ImageFolder

dataset = ImageFolder(DATA_DIR)

class_names = dataset.classes
num_classes = len(class_names)

print("Total images:", len(dataset))
print("Total classes:", num_classes)

for i, name in enumerate(class_names):
    print(i, name)

Total images: 20638
Total classes: 15
0 Pepper__bell___Bacterial_spot
1 Pepper__bell___healthy
2 Potato___Early_blight
3 Potato___Late_blight
4 Potato___healthy
5 Tomato_Bacterial_spot
6 Tomato_Early_blight
7 Tomato_Late_blight
8 Tomato_Leaf_Mold
9 Tomato_Septoria_leaf_spot
10 Tomato_Spider_mites_Two_spotted_spider_mite
11 Tomato__Target_Spot
12 Tomato__Tomato_YellowLeaf__Curl_Virus
13 Tomato__Tomato_mosaic_virus
14 Tomato_healthy


In [14]:

from torchvision import transforms, datasets
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split

# Image preprocessing
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Load images with separate transforms
train_full = datasets.ImageFolder(
    DATA_DIR, transform=train_transform
)

val_full = datasets.ImageFolder(
    DATA_DIR, transform=val_transform
)

class_names = train_full.classes
num_classes = len(class_names)

# Stratified 80/20 split
indices = np.arange(len(train_full))
targets = np.array(train_full.targets)

train_indices, val_indices = train_test_split(
    indices,
    test_size=0.2,
    random_state=42,
    stratify=targets
)

train_dataset = Subset(train_full, train_indices)
val_dataset = Subset(val_full, val_indices)

print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Number of classes:", num_classes)


Training images: 16510
Validation images: 4128
Number of classes: 15


In [15]:

BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Device:", "CUDA" if torch.cuda.is_available() else "CPU")


Image batch shape: torch.Size([16, 3, 224, 224])
Label batch shape: torch.Size([16])
Device: CUDA


In [16]:

import torch.nn as nn
from torchvision import models
from torchvision.models import ResNet50_Weights

# Load ResNet50 pretrained on ImageNet
weights = ResNet50_Weights.DEFAULT
model = models.resnet50(weights=weights)

# Freeze all pretrained parameters
for param in model.parameters():
    param.requires_grad = False

# Replace the original 1000-class ImageNet output layer
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, num_classes)

# Move model to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print("Device:", device)
print("Number of disease classes:", num_classes)
print(model.fc)


Device: cuda
Number of disease classes: 15
Linear(in_features=2048, out_features=15, bias=True)


In [17]:

import torch.optim as optim

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.fc.parameters(),
    lr=0.001
)

EPOCHS = 5

print("Loss function:", criterion)
print("Optimizer:", optimizer)
print("Epochs:", EPOCHS)


Loss function: CrossEntropyLoss()
Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)
Epochs: 5


In [18]:

import copy

best_val_accuracy = 0.0
best_state = copy.deepcopy(model.state_dict())

train_losses = []
val_accuracies = []

for epoch in range(EPOCHS):

    # Training phase
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(train_dataset)
    train_losses.append(epoch_loss)

    # Validation phase
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            predictions = outputs.argmax(dim=1)

            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    val_accuracy = correct / total
    val_accuracies.append(val_accuracy)

    print(
        f"Epoch {epoch + 1}/{EPOCHS} | "
        f"Loss: {epoch_loss:.4f} | "
        f"Validation Accuracy: {val_accuracy:.4f}"
    )

    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        best_state = copy.deepcopy(model.state_dict())

# Restore the best checkpoint from training
model.load_state_dict(best_state)

print(f"\nBest validation accuracy: {best_val_accuracy:.4f}")


Epoch 1/5 | Loss: 0.8081 | Validation Accuracy: 0.9111
Epoch 2/5 | Loss: 0.3545 | Validation Accuracy: 0.9317
Epoch 3/5 | Loss: 0.2733 | Validation Accuracy: 0.9394
Epoch 4/5 | Loss: 0.2340 | Validation Accuracy: 0.9491
Epoch 5/5 | Loss: 0.2023 | Validation Accuracy: 0.9501

Best validation accuracy: 0.9501


In [19]:
from sklearn.metrics import classification_report

model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)

        outputs = model(images)
        predictions = outputs.argmax(dim=1)

        all_predictions.extend(predictions.cpu().numpy())
        all_labels.extend(labels.numpy())

print(classification_report(
    all_labels,
    all_predictions,
    target_names=class_names,
    digits=4,
    zero_division=0
))

                                             precision    recall  f1-score   support

              Pepper__bell___Bacterial_spot     0.9792    0.9447    0.9616       199
                     Pepper__bell___healthy     0.9609    0.9966    0.9784       296
                      Potato___Early_blight     0.9756    1.0000    0.9877       200
                       Potato___Late_blight     0.9310    0.9450    0.9380       200
                           Potato___healthy     0.9231    0.8000    0.8571        30
                      Tomato_Bacterial_spot     0.9605    0.9695    0.9650       426
                        Tomato_Early_blight     0.9244    0.7950    0.8548       200
                         Tomato_Late_blight     0.9112    0.9398    0.9253       382
                           Tomato_Leaf_Mold     0.9465    0.9316    0.9390       190
                  Tomato_Septoria_leaf_spot     0.9233    0.9520    0.9374       354
Tomato_Spider_mites_Two_spotted_spider_mite     0.9377    0.9433

In [20]:
import torch
import os

checkpoint = {
    "model_name": "resnet50",
    "num_classes": num_classes,
    "class_names": class_names,
    "model_state_dict": model.state_dict(),
    "best_val_accuracy": best_val_accuracy,
    "image_size": 224,
    "mean": [0.485, 0.456, 0.406],
    "std": [0.229, 0.224, 0.225]
}

save_path = r"D:\Desktop\ml_lab\plant_disease_resnet50.pth"

torch.save(checkpoint, save_path)

print("Model saved successfully!")
print("Location:", save_path)
print("Validation accuracy:", f"{best_val_accuracy:.2%}")

Model saved successfully!
Location: D:\Desktop\ml_lab\plant_disease_resnet50.pth
Validation accuracy: 95.01%
